# DCL4 (A) / DRB2 (B) / DRB4 (C) / ds-RNA (D,E) Domain Contact Analysis -- **synthetic-template run**

**Kernel:** `abcfold-drbs-notebook` (`envs/notebook.yaml`) -- install once:
```
conda env create -f envs/notebook.yaml
conda activate abcfold-drbs-notebook
python -m ipykernel install --user --name abcfold-drbs-notebook
```

This is `rna_ds_dcl4_drb2_drb4_domain_analysis.ipynb` re-pointed at the
**synthetic-template re-run** `rna_ds_dcl4_drb2_drb4_synthtmpl_01`
(`RESULTS_DIR` in the next cell). That run is AlphaFold3 + OpenFold3 only,
with a top-ipTM **Boltz** DRB2+DRB4 conformation (from the DCL4-free
`rna_ds_drb2_drb4` over-folding survivors) injected into the full complex as
an AF3-dialect per-chain custom template on DRB2 and DRB4 -- see
`configs/rna_ds_dcl4_drb2_drb4_synthtmpl_01.yaml`'s `synthetic_template:`
block, `scripts/select_synthetic_templates.py` /
`scripts/inject_synthetic_template.py`, and commit `9c886f6`.

**Why this run exists.** In the baseline `rna_ds_dcl4_drb2_drb4` both AF3 and
OpenFold3 fold the DRB2/DRB4 disordered tails into ~60% helix, so ~0 models
survive the non-MoRF over-folding filter. With the Boltz template injected,
**AlphaFold3's DRB2 disordered tail (B 189-434) drops from ~66% helix / max
run 24 to ~4% / 4**, and ~38 of 100 AF3 models now pass a quick-look
over-folding filter (DRB4 only partially rescued, run ~13; OpenFold3
unchanged). Numbers: `results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/`
`dssp_summary.csv`, `overfolding_helix_stats.csv`,
`overfolding_survivors_quicklook.tsv`. This notebook is the PLIP
*domain-contact* view of that ensemble -- run it and diff the heatmaps
against the baseline notebook.

> **Provenance note:** the cluster run of `synthtmpl_01` was cut short by an
> IFB inode-quota failure at OpenFold3 seed 18/20, so this ensemble is
> AF3 100/100 + OpenFold3 90/100 (190 models), recovered and postprocessed
> locally. `synthtmpl_02` / `synthtmpl_03` (2nd/3rd-ranked Boltz templates)
> are a separate re-run.

---

PLIP contacts from
`results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/all_selected_summary.csv`,
produced by `workflows/postprocessing/Snakefile` (stage 3g `run_plip` + 3h
`aggregate`).

**Scope limitation, important:**
`configs/rna_ds_dcl4_drb2_drb4_synthtmpl_01.yaml`'s `plip.chains` is
`[['A'], ['B', 'C', 'D', 'E']]` (copied verbatim from
`rna_ds_dcl4_drb2_drb4.yaml`) -- receptor = **DCL4 only**, ligand = DRB2 +
DRB4 + both RNA strands combined. PLIP only reports receptor-vs-ligand
contacts, never ligand-vs-ligand ones, so this single PLIP pass can only ever
surface **DCL4-vs-X** interfaces (`reschain` is always `A`). It structurally
cannot see DRB2-DRB4, DRB2-RNA or DRB4-RNA contacts -- those need the
dedicated passes (`plip_rna_ligands:` / `plip_drb2_drb4:`), folded in below.
So an *absence* of those interfaces here is a config artefact, not a finding.

**RNA-ligand pass:**
`results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/all_selected_summary_rna_ligands.csv`
(receptor = each protein chain, ligand = RNA strands D/E). The load cell
folds in **only its `reschain == "A"` rows**, adding the **DCL4 x RNA(D)** and
**DCL4 x RNA(E)** couples. **DRB2-DRB4** has its own third pass
(`all_selected_summary_drb2_drb4.csv`), loaded in the dedicated section near
the end.

**Data provenance quirk** (see `drb2_drb4_domain_analysis.ipynb` for the full
explanation): `scripts/aggregate_summaries.py`'s "replica" column is actually
this pipeline's pose **cluster** number, and "model" is the staged **fname**.
Renamed below for clarity.

Pose clusters come from `scripts/pose_cluster_anchor.py`'s rigid-anchor (DCL4)
Kabsch + hierarchical RMSD clustering
(`results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/pose_clusters.csv`); this
notebook stratifies **PLIP domain contacts** by that cluster label via
`selected_models.csv`, it does not re-derive clusters.

An energy-based filter (robust MAD approach, same as
`drb2_drb4_domain_analysis.ipynb`) removes numerically-unconverged
minimizations before any analysis below. (This run has no RosettaFold3, so
the whole-backend RF3 drop the baseline notebook describes does not apply --
expect AlphaFold3 + OpenFold3 throughout.)

Domain boundaries (1-based inclusive) -- **identical to
`rna_ds_dcl4_drb2_drb4`**, sequences are byte-for-byte the same. DRB2's
dsRBD2/disordered boundary is the corrected 87-188 / 189-434 call (residues
156-188 are genuinely folded -- see
`notebooks/drb2_drb4_domain_analysis.ipynb`'s fold-upon-binding
investigation).

**DCL4 (chain A)**

| DCL4 domain | Residues |
|---|---|
| helicase | 131-629 |
| DUF283 | 651-753 |
| platform | 754-931 |
| PAZ | 932-1054 |
| connector | 1055-1082 |
| RNase_IIIa | 1083-1251 |
| RNase_IIIb | 1292-1436 |
| dsRBD1 | 1462-1528 |
| linker | 1529-1620 |
| dsRBD2 | 1621-1697 |

**DRB2 (chain B)**

| DRB2 domain | Residues |
|---|---|
| dsRBD1 | 1-70 |
| linker | 71-86 |
| dsRBD2 | 87-188 |
| disordered | 189-434 |

**DRB4 (chain C)**

| DRB4 domain | Residues |
|---|---|
| dsRBD1 | 4-73 |
| linker | 74-81 |
| dsRBD2 | 82-150 |
| disordered | 151-291 |
| cryoEM_domain | 292-355 |

**ds-RNA (chains D/E)** -- nucleotide position only (D = 57 nt sense, E = 55 nt
antisense).


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

ROOT = Path("..")
RESULTS_DIR = ROOT / "results" / "rna_ds_dcl4_drb2_drb4_synthtmpl_01"
RECEPTOR_CHAIN, RECEPTOR_NAME = "A", "DCL4"
LIGAND_CHAIN_NAMES = {"B": "DRB2", "C": "DRB4", "D": "RNA(D)", "E": "RNA(E)"}
FIGURES_DIR = RESULTS_DIR / "figures" / "domain_analysis"

TEMPLATE = "plotly_white"
CLUSTER_PALETTE = px.colors.qualitative.Set1
BACKEND_PALETTE = px.colors.qualitative.Set2

def out_path(subdir, filename):
    out_dir = FIGURES_DIR / subdir
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / filename

def save_fig(fig, filename, subdir=""):
    out = out_path(subdir, filename)
    fig.write_html(out, include_plotlyjs="cdn")
    print(f"Saved: {out}")
    fig.show()


## Load data

In [2]:
csv_path = RESULTS_DIR / "all_selected_summary.csv"
df = pd.read_csv(csv_path)

# The dedicated RNA-ligand PLIP pass (configs' `plip_rna_ligands:` block -- receptor =
# each protein chain, ligand = RNA strands D/E) landed in its own summary file. Fold in
# ONLY its DCL4-as-receptor rows (reschain == RECEPTOR_CHAIN) so this notebook's "DCL4 is
# the fixed receptor" invariant still holds -- that adds the DCL4 x RNA(D) / DCL4 x RNA(E)
# couples that all_selected_summary.csv structurally cannot contain. Its DRB2-RNA / DRB4-RNA
# rows are deliberately left out here.
rna_csv = RESULTS_DIR / "all_selected_summary_rna_ligands.csv"
if rna_csv.exists():
    rna_df = pd.read_csv(rna_csv)
    rna_df = rna_df[rna_df["reschain"] == RECEPTOR_CHAIN].copy()
    print(f"{rna_csv.name}: +{len(rna_df)} DCL4-vs-RNA contact rows "
          f"({rna_df['reschain_lig'].value_counts().to_dict()})")
    df = pd.concat([df, rna_df], ignore_index=True)
else:
    print(f"{rna_csv.name} not found -- DCL4 x RNA couples stay empty "
          f"(run the plip_rna_ligands pass to populate them)")

df = df.rename(columns={"replica": "cluster", "model": "fname"})
df["cluster"] = df["cluster"].astype(int)

sel = pd.read_csv(RESULTS_DIR / "selected_models.csv")
sel["fname"] = sel["staged_cif"].apply(lambda p: Path(p).stem)
sel = sel[["fname", "cluster", "backend", "seed", "sample_index", "ranking_score", "ptm", "iptm"]]

df = df.merge(sel, on=["fname", "cluster"], how="left", validate="many_to_one")
n_missing_meta = df["backend"].isna().sum()
if n_missing_meta:
    print(f"WARNING: {n_missing_meta} contact rows have no matching selected_models.csv "
          f"entry (stale/partial aggregate CSV -- re-run after the full postprocessing "
          f"run finishes)")

n_models = df.groupby(["cluster", "fname"]).ngroups
print(f"{len(df)} contact rows total, {df['cluster'].nunique()} pose cluster(s), {n_models} model(s), "
      f"before the energy filter below")
print("contact rows per receptor-ligand chain pair:")
print(df.groupby(["reschain", "reschain_lig"]).size().rename("rows").to_frame())

all_selected_summary_rna_ligands.csv: +23540 DCL4-vs-RNA contact rows ({'D': 11820, 'E': 11720})
46326 contact rows total, 2 pose cluster(s), 190 model(s), before the energy filter below
contact rows per receptor-ligand chain pair:
                        rows
reschain reschain_lig       
A        B             12325
         C             10461
         D             11820
         E             11720


## Filter out numerically-unconverged structures

Same robust (median / median-absolute-deviation) modified z-score approach
as `drb2_drb4_domain_analysis.ipynb` -- applied here, right after loading,
so every section below (per-couple heatmaps, per-cluster, per-backend) works
from the cleaned set.


In [3]:
def read_final_energy(energy_csv_path):
    if not energy_csv_path.exists():
        return np.nan
    last_energy = np.nan
    with open(energy_csv_path) as fh:
        next(fh, None)
        for line in fh:
            _, e = line.strip().split(",")
            last_energy = float(e)
    return last_energy

energy_rows = []
for pdb_path in sorted(RESULTS_DIR.glob("minimized/*/*/*.pdb")):
    if pdb_path.stem.endswith(("_fixed", "_amber")):
        continue
    cluster = int(pdb_path.parent.parent.name)
    energy_csv = pdb_path.with_name(pdb_path.stem + "_energy.csv")
    energy_rows.append({"fname": pdb_path.stem, "cluster": cluster,
                         "final_energy": read_final_energy(energy_csv)})

energy_df = pd.DataFrame(energy_rows).dropna(subset=["final_energy"])

MOD_Z_THRESHOLD = 3.5
pooled_median = energy_df["final_energy"].median()
pooled_mad = (energy_df["final_energy"] - pooled_median).abs().median()
energy_df["energy_mod_z"] = 0.6745 * (energy_df["final_energy"] - pooled_median) / pooled_mad
energy_df["energy_ok"] = energy_df["energy_mod_z"].abs() <= MOD_Z_THRESHOLD

flagged = energy_df.loc[~energy_df["energy_ok"]].merge(
    sel[["fname", "cluster", "backend"]], on=["fname", "cluster"], how="left"
)
print(f"Pooled final-energy median={pooled_median:,.0f} kJ/mol, MAD={pooled_mad:,.0f}")
print(f"Flagged {len(flagged)} / {len(energy_df)} minimized model(s) as numerically "
      f"unconverged (|modified z-score| > {MOD_Z_THRESHOLD}), by backend:")
display(flagged.groupby("backend").size().rename("n_excluded").to_frame())


Pooled final-energy median=-158,932 kJ/mol, MAD=4,084
Flagged 9 / 180 minimized model(s) as numerically unconverged (|modified z-score| > 3.5), by backend:


,n_excluded
backend,
alphafold3,9


In [4]:
good_pairs = set(zip(
    energy_df.loc[energy_df["energy_ok"], "cluster"],
    energy_df.loc[energy_df["energy_ok"], "fname"],
))
df_keys = pd.MultiIndex.from_arrays([df["cluster"], df["fname"]])
n_models_before = df.groupby(["cluster", "fname"]).ngroups
df = df[df_keys.isin(good_pairs)].copy()
n_models_after = df.groupby(["cluster", "fname"]).ngroups

print(f"{n_models_after} / {n_models_before} model(s) kept after the energy filter")
print(df.groupby(["cluster", "backend"]).apply(lambda g: g[["fname"]].drop_duplicates().shape[0], include_groups=False)
      .rename("n_models").to_frame())


171 / 190 model(s) kept after the energy filter
                    n_models
cluster backend             
1       alphafold3        70
        openfold3         76
2       alphafold3        14
        openfold3         11


## Domain definitions

In [5]:
DCL4_DOMAINS = [
    ("helicase", 131, 629),
    ("DUF283", 651, 753),
    ("platform", 754, 931),
    ("PAZ", 932, 1054),
    ("connector", 1055, 1082),
    ("RNase_IIIa", 1083, 1251),
    ("RNase_IIIb", 1292, 1436),
    ("dsRBD1", 1462, 1528),
    ("linker", 1529, 1620),
    ("dsRBD2", 1621, 1697),
]

DRB2_DOMAINS = [
    ("dsRBD1", 1, 70),
    ("linker", 71, 86),
    ("dsRBD2", 87, 188),   # corrected from 87-155 -- see intro cell
    ("disordered", 189, 434),   # corrected from 156-434
]

DRB4_DOMAINS = [
    ("dsRBD1", 4, 73),
    ("linker", 74, 81),
    ("dsRBD2", 82, 150),
    ("disordered", 151, 291),
    ("cryoEM_domain", 292, 355),
]

def make_domain_mapper(domain_ranges):
    """domain_ranges: list of (label, start, end), inclusive on both ends.
    Returns a function mapping a pandas Series of residue numbers to domain
    labels; residues outside every range become NaN (reported separately)."""
    intervals = pd.IntervalIndex.from_tuples(
        [(start, end) for _, start, end in domain_ranges], closed="both"
    )
    labels = [label for label, _, _ in domain_ranges]

    def mapper(resnr_series):
        idx = intervals.get_indexer(resnr_series.astype(float))
        return pd.Series(
            [labels[i] if i != -1 else pd.NA for i in idx],
            index=resnr_series.index, dtype="object",
        )
    return mapper

DCL4_LABELS = [l for l, _, _ in DCL4_DOMAINS]
DRB2_LABELS = [l for l, _, _ in DRB2_DOMAINS]
DRB4_LABELS = [l for l, _, _ in DRB4_DOMAINS]

dcl4_mapper = make_domain_mapper(DCL4_DOMAINS)
LIGAND_DOMAINS = {"B": DRB2_DOMAINS, "C": DRB4_DOMAINS}
LIGAND_LABELS  = {"B": DRB2_LABELS, "C": DRB4_LABELS}
LIGAND_MAPPERS = {c: make_domain_mapper(d) for c, d in LIGAND_DOMAINS.items()}

df["dcl4_domain"] = dcl4_mapper(df["resnr"])

# IMPORTANT, previously-unnoticed fix: this complex needs fix_pdb (RNA-
# containing -> pdb4amber + PDBFixer), which renumbers the WHOLE complex
# continuously across every chain rather than restarting each chain at 1 --
# so PLIP's resnr_lig for DRB2/DRB4 was never in their own 1-based numbering
# our domain tables assume. Verified directly from the fixed PDBs' own
# per-chain residue ranges (identical across a 15-model random sample
# spanning every backend/seed): chain A (DCL4) 1-1701, chain B (DRB2)
# 1702-2135 (434 residues), chain C (DRB4) 2136-2490 (355 residues), chain D
# (RNA sense) 2491-2547, chain E (RNA antisense) 2548-2602. Without this
# correction, .dropna() below silently discarded essentially every DRB2/DRB4
# contact row (resnr_lig never falls inside [1,434]/[1,355]) -- every
# DCL4xDRB2/DCL4xDRB4 heatmap in every earlier version of this notebook was
# silently empty.
CHAIN_OFFSET = {"B": 1701, "C": 2135, "D": 2490, "E": 2547}
df["resnr_lig_raw"] = df["resnr_lig"]
for chain, offset in CHAIN_OFFSET.items():
    mask = df["reschain_lig"] == chain
    df.loc[mask, "resnr_lig"] = df.loc[mask, "resnr_lig"] - offset

for chain, expected_len in [("B", 434), ("C", 355), ("D", 57), ("E", 55)]:
    sub = df[df["reschain_lig"] == chain]
    if sub.empty:
        continue
    bad = sub[~sub["resnr_lig"].between(1, expected_len)]
    if len(bad):
        print(f"WARNING: {len(bad)} chain-{chain} rows fall outside [1,{expected_len}] "
              f"after offset correction -- re-verify CHAIN_OFFSET['{chain}'].")
    else:
        print(f"chain {chain}: offset {CHAIN_OFFSET[chain]} verified OK "
              f"(all {len(sub)} rows land in [1,{expected_len}])")

df["ligand_domain"] = pd.NA
for chain, mapper in LIGAND_MAPPERS.items():
    mask = df["reschain_lig"] == chain
    df.loc[mask, "ligand_domain"] = mapper(df.loc[mask, "resnr_lig"])

n_unmapped_r = df["dcl4_domain"].isna().sum()
print(f"Unmapped DCL4 residues (outside any domain): {n_unmapped_r} ({100*n_unmapped_r/len(df):.1f}%)")
df[["resnr", "dcl4_domain", "reschain_lig", "resnr_lig_raw", "resnr_lig", "ligand_domain"]].head(5)


chain B: offset 1701 verified OK (all 10328 rows land in [1,434])
chain C: offset 2135 verified OK (all 9598 rows land in [1,355])
chain D: offset 2490 verified OK (all 11126 rows land in [1,57])
chain E: offset 2547 verified OK (all 11018 rows land in [1,55])
Unmapped DCL4 residues (outside any domain): 3721 (8.8%)


,resnr,dcl4_domain,reschain_lig,resnr_lig_raw,resnr_lig,ligand_domain
0,24,<NA>,B,1733,32,dsRBD1
1,28,<NA>,B,1750,49,dsRBD1
2,403,helicase,B,2134,433,disordered
3,410,helicase,B,2130,429,disordered
4,413,helicase,B,2130,429,disordered


## Interactor couples in this dataset

Every heatmap in this notebook is one **couple**: DCL4 (fixed receptor) vs.
one specific partner. Four couples are structurally possible given the
`--chains` scope limitation above (DCL4-DRB2, DCL4-DRB4, DCL4-RNA(D),
DCL4-RNA(E)) -- which of them actually have contact data is shown below,
before building any heatmaps, so an empty couple later isn't a surprise.


In [6]:
COUPLES = ["B", "C", "D", "E"]  # DRB2, DRB4, RNA(D), RNA(E)

print("Contact rows per couple (DCL4 vs. partner):")
for c in COUPLES:
    n = (df["reschain_lig"] == c).sum()
    n_models_c = df[df["reschain_lig"] == c][["cluster", "fname"]].drop_duplicates().shape[0]
    status = "POPULATED" if n else "EMPTY -- no contacts observed"
    print(f"  DCL4 x {LIGAND_CHAIN_NAMES[c]:8s}: {n:6d} contact rows across {n_models_c:3d} model(s) -- {status}")


Contact rows per couple (DCL4 vs. partner):
  DCL4 x DRB2    :  10328 contact rows across 169 model(s) -- POPULATED
  DCL4 x DRB4    :   9598 contact rows across 170 model(s) -- POPULATED
  DCL4 x RNA(D)  :  11126 contact rows across 171 model(s) -- POPULATED
  DCL4 x RNA(E)  :  11018 contact rows across 171 model(s) -- POPULATED


In [7]:
def domain_pair_heatmap(data, ligand_chain, title, filename, n_models_norm, subdir="domain_contacts"):
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    labels = LIGAND_LABELS[ligand_chain]
    sub = data[(data["reschain_lig"] == ligand_chain)].dropna(subset=["dcl4_domain", "ligand_domain"])
    if sub.empty:
        print(f"No DCL4-{ligand_name} contacts in this subset -- skipping '{title}'.")
        return None

    ct = (
        sub.groupby(["dcl4_domain", "ligand_domain"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(index=DCL4_LABELS, columns=labels, fill_value=0)
    )
    rate = ct / n_models_norm if n_models_norm else ct

    fig = go.Figure(go.Heatmap(
        z=rate.values, x=labels, y=DCL4_LABELS,
        colorscale="YlOrRd",
        text=[[f"{v:.2f}" if v > 0 else "" for v in row] for row in rate.values],
        texttemplate="%{text}", textfont=dict(size=9),
        hovertemplate=f"DCL4 domain: %{{y}}<br>{ligand_name} domain: %{{x}}<br>%{{z:.3f}} contacts/model<extra></extra>",
        colorbar=dict(title="Mean<br>contacts/<br>model"),
    ))
    fig.update_layout(
        title=title, xaxis_title=f"{ligand_name} domain", yaxis_title="DCL4 domain",
        yaxis=dict(autorange="reversed"), template=TEMPLATE,
        width=max(500, len(labels) * 100), height=max(500, len(DCL4_LABELS) * 40),
    )
    save_fig(fig, filename, subdir)
    return ct

def rna_contact_heatmap(data, strand, n_models_norm, title=None, filename=None, subdir="domain_contacts"):
    sub = data[(data["reschain_lig"] == strand)].dropna(subset=["dcl4_domain"])
    if sub.empty:
        print(f"No DCL4-RNA({strand}) contacts in this subset -- skipping.")
        return None

    nt_positions = sorted(sub["resnr_lig"].dropna().unique())
    ct = (
        sub.groupby(["resnr_lig", "dcl4_domain"], observed=True)
        .size()
        .unstack(fill_value=0)
        .reindex(index=nt_positions, columns=DCL4_LABELS, fill_value=0)
    )
    rate = ct / n_models_norm if n_models_norm else ct

    fig = go.Figure(go.Heatmap(
        z=rate.values, x=DCL4_LABELS, y=[str(p) for p in nt_positions],
        colorscale="YlOrRd",
        hovertemplate=(f"DCL4 domain: %{{x}}<br>RNA nt (strand {strand}): %{{y}}"
                        "<br>%{z:.3f} contacts/model<extra></extra>"),
        colorbar=dict(title="Contacts<br>/ model"),
    ))
    fig.update_layout(
        title=title or f"DCL4 x RNA strand {strand} contact rate",
        xaxis_title="DCL4 domain", yaxis_title=f"RNA nt position (strand {strand})",
        yaxis=dict(autorange="reversed", tickfont=dict(size=8)),
        template=TEMPLATE, width=900, height=max(400, len(nt_positions) * 14),
    )
    save_fig(fig, filename or f"dcl4_rna_{strand}_heatmap.html", subdir)
    return ct

def couple_heatmap(data, ligand_chain, n_models_norm, title, filename, subdir="domain_contacts"):
    """Dispatch to the right heatmap shape for this couple -- domain x domain
    for protein partners (B, C), nt-position x domain for RNA strands (D, E).
    One function so every section below can loop over COUPLES uniformly."""
    if ligand_chain in ("B", "C"):
        return domain_pair_heatmap(data, ligand_chain, title, filename, n_models_norm, subdir)
    return rna_contact_heatmap(data, ligand_chain, n_models_norm, title, filename, subdir)


## One heatmap per couple (all clusters, all backends pooled)

In [8]:
n_models_total = df.groupby(["cluster", "fname"]).ngroups
ct_all = {}
for c in COUPLES:
    ct_all[c] = couple_heatmap(
        df, c, n_models_total,
        f"DCL4 x {LIGAND_CHAIN_NAMES[c]} -- all clusters, all backends (n={n_models_total} models)",
        f"dcl4_{LIGAND_CHAIN_NAMES[c].lower()}_heatmap_pooled.html",
    )


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/domain_contacts/dcl4_drb2_heatmap_pooled.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/domain_contacts/dcl4_drb4_heatmap_pooled.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/domain_contacts/dcl4_rna(d)_heatmap_pooled.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/domain_contacts/dcl4_rna(e)_heatmap_pooled.html


## Interaction-type breakdown (all clusters pooled)

In [9]:
print("Interaction type counts:")
display(df["interaction_type"].value_counts().rename("count").to_frame())

print("\nInteraction type by receptor-ligand chain pair:")
display(
    df.groupby(["reschain", "reschain_lig", "interaction_type"], observed=True)
    .size().rename("count").reset_index().sort_values("count", ascending=False).head(20)
)


Interaction type counts:


,count
interaction_type,
hydrogen_bonds,22807
salt_bridges,12259
hydrophobic_interactions,6571
pi-cation_interactions,357
pi-stacking,76



Interaction type by receptor-ligand chain pair:


,reschain,reschain_lig,interaction_type,count
15,A,E,hydrogen_bonds,6358
10,A,D,hydrogen_bonds,6338
0,A,B,hydrogen_bonds,5271
5,A,C,hydrogen_bonds,4840
14,A,D,salt_bridges,4691
19,A,E,salt_bridges,4579
1,A,B,hydrophobic_interactions,3399
6,A,C,hydrophobic_interactions,3134
4,A,B,salt_bridges,1543
9,A,C,salt_bridges,1446


## Per-cluster breakdown

One section per pose cluster (`cluster` column, from `pose_cluster_anchor.py`'s
rigid-anchor Kabsch + hierarchical RMSD clustering, loaded via
`selected_models.csv`) -- each with its own couple heatmaps, so a difference
in domain-domain contacts between clusters shows up directly rather than
being averaged away in the pooled view above.


In [10]:
clusters = sorted(df["cluster"].unique())
cluster_n_models = df.groupby("cluster").apply(lambda g: g["fname"].nunique(), include_groups=False)
print(f"{len(clusters)} pose cluster(s): " +
      ", ".join(f"cluster {c} (n={cluster_n_models[c]} models)" for c in clusters))


2 pose cluster(s): cluster 1 (n=146 models), cluster 2 (n=25 models)


### Cluster 1

In [11]:
c = 1
sub_c = df[df["cluster"] == c]
n_c = cluster_n_models.get(c, 0)
print(f"cluster {c}: {n_c} model(s)")

for lig in COUPLES:
    couple_heatmap(
        sub_c, lig, n_c,
        f"DCL4 x {LIGAND_CHAIN_NAMES[lig]} -- cluster {c} (n={n_c} models)",
        f"dcl4_{LIGAND_CHAIN_NAMES[lig].lower()}_heatmap_cluster1.html",
        subdir="per_cluster",
    )


cluster 1: 146 model(s)
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_drb2_heatmap_cluster1.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_drb4_heatmap_cluster1.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_rna(d)_heatmap_cluster1.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_rna(e)_heatmap_cluster1.html


In [12]:
print("Interaction type counts, cluster 1:")
display(df[df["cluster"] == 1]["interaction_type"].value_counts().rename("count").to_frame())


Interaction type counts, cluster 1:


,count
interaction_type,
hydrogen_bonds,19532
salt_bridges,10418
hydrophobic_interactions,5745
pi-cation_interactions,301
pi-stacking,68


### Cluster 2

In [13]:
c = 2
sub_c = df[df["cluster"] == c]
n_c = cluster_n_models.get(c, 0)
print(f"cluster {c}: {n_c} model(s)")

for lig in COUPLES:
    couple_heatmap(
        sub_c, lig, n_c,
        f"DCL4 x {LIGAND_CHAIN_NAMES[lig]} -- cluster {c} (n={n_c} models)",
        f"dcl4_{LIGAND_CHAIN_NAMES[lig].lower()}_heatmap_cluster2.html",
        subdir="per_cluster",
    )


cluster 2: 25 model(s)
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_drb2_heatmap_cluster2.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_drb4_heatmap_cluster2.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_rna(d)_heatmap_cluster2.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_rna(e)_heatmap_cluster2.html


In [14]:
print("Interaction type counts, cluster 2:")
display(df[df["cluster"] == 2]["interaction_type"].value_counts().rename("count").to_frame())


Interaction type counts, cluster 2:


,count
interaction_type,
hydrogen_bonds,3275
salt_bridges,1841
hydrophobic_interactions,826
pi-cation_interactions,56
pi-stacking,8


## Per-backend breakdown

For each couple, a single panel figure with one heatmap per backend (shared
color scale within the panel) -- mirrors `drb2_drb4_domain_analysis.ipynb`'s
per-backend section. Only alphafold3 and openfold3 have any energy-filtered
models left (rosettafold3 dropped out entirely, see the filter section
above), so these panels are 2 columns wide rather than the up-to-6 in the
binary complex notebook.


In [15]:
backends = sorted(df["backend"].dropna().unique())
backend_n_models = df.groupby("backend").apply(lambda g: g[["fname","cluster"]].drop_duplicates().shape[0], include_groups=False)
print(f"{len(backends)} backend(s) with energy-filtered models: " +
      ", ".join(f"{b} (n={backend_n_models[b]} models)" for b in backends))


2 backend(s) with energy-filtered models: alphafold3 (n=84 models), openfold3 (n=87 models)


In [16]:
def couple_heatmap_matrix(data, ligand_chain, n_models_norm):
    """Same content as couple_heatmap() but returns (labels, y_labels, rate matrix)
    without plotting -- used to build multi-panel (per-backend / per-cluster)
    comparisons without generating (and discarding) N individual figures first."""
    if ligand_chain in ("B", "C"):
        labels = LIGAND_LABELS[ligand_chain]
        sub = data[(data["reschain_lig"] == ligand_chain)].dropna(subset=["dcl4_domain", "ligand_domain"])
        if sub.empty:
            return None
        ct = (sub.groupby(["dcl4_domain", "ligand_domain"], observed=True).size()
              .unstack(fill_value=0).reindex(index=DCL4_LABELS, columns=labels, fill_value=0))
        y_labels = DCL4_LABELS
    else:
        sub = data[(data["reschain_lig"] == ligand_chain)].dropna(subset=["dcl4_domain"])
        if sub.empty:
            return None
        nt_positions = sorted(sub["resnr_lig"].dropna().unique())
        ct = (sub.groupby(["resnr_lig", "dcl4_domain"], observed=True).size()
              .unstack(fill_value=0).reindex(index=nt_positions, columns=DCL4_LABELS, fill_value=0))
        labels, y_labels = DCL4_LABELS, [str(p) for p in nt_positions]
    return labels, y_labels, (ct / n_models_norm if n_models_norm else ct)

def backend_panel_for_couple(ligand_chain):
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    per_backend = {}
    for b in backends:
        res = couple_heatmap_matrix(df[df["backend"] == b], ligand_chain, backend_n_models[b])
        if res is not None:
            per_backend[b] = res
    if not per_backend:
        print(f"No DCL4-{ligand_name} contacts for any backend -- skipping panel.")
        return

    present = list(per_backend.keys())
    zmax = max(rate.values.max() for _, _, rate in per_backend.values())
    ncols = min(3, len(present))
    nrows = -(-len(present) // ncols)

    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[f"{b} (n={backend_n_models[b]})" for b in present],
        shared_yaxes=True,
    )
    for i, b in enumerate(present):
        labels, y_labels, rate = per_backend[b]
        row, col = i // ncols + 1, i % ncols + 1
        fig.add_trace(
            go.Heatmap(
                z=rate.values, x=labels, y=y_labels, colorscale="YlOrRd",
                zmin=0, zmax=zmax, showscale=(b == present[-1]),
                text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in rate.values],
                texttemplate="%{text}", textfont=dict(size=8),
                hovertemplate=f"{b}<br>DCL4: %{{y}}<br>{ligand_name}: %{{x}}<br>%{{z:.3f}} contacts/model<extra></extra>",
                colorbar=dict(title="Mean<br>contacts/<br>model"),
            ),
            row=row, col=col,
        )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        title=f"DCL4 x {ligand_name} domain contact pairs by backend",
        template=TEMPLATE, width=max(700, 380 * ncols), height=max(460, 40 * len(y_labels if ligand_chain in ('D','E') else DCL4_LABELS)),
    )
    save_fig(fig, f"dcl4_{ligand_name.lower()}_heatmap_by_backend.html", "per_backend")

for c in COUPLES:
    backend_panel_for_couple(c)


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_backend/dcl4_drb2_heatmap_by_backend.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_backend/dcl4_drb4_heatmap_by_backend.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_backend/dcl4_rna(d)_heatmap_by_backend.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_backend/dcl4_rna(e)_heatmap_by_backend.html


## Export per-cluster / per-couple domain contact tables

In [17]:
for c in clusters:
    sub_c = df[df["cluster"] == c]
    n_c = cluster_n_models[c]
    for lig in COUPLES:
        ct = couple_heatmap_matrix(sub_c, lig, n_c)
        if ct is None:
            continue
        _, _, rate = ct
        out_csv = out_path("per_cluster", f"dcl4_{LIGAND_CHAIN_NAMES[lig].lower()}_domain_pair_counts_cluster{c}.csv")
        rate.to_csv(out_csv)
        print(f"Saved: {out_csv}")


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_drb2_domain_pair_counts_cluster1.csv


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_drb4_domain_pair_counts_cluster1.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_rna(d)_domain_pair_counts_cluster1.csv


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_rna(e)_domain_pair_counts_cluster1.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_drb2_domain_pair_counts_cluster2.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_drb4_domain_pair_counts_cluster2.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_rna(d)_domain_pair_counts_cluster2.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_cluster/dcl4_rna(e)_domain_pair_counts_cluster2.csv


---

# Manual pose clustering -- 4 quadrants of the PCA embedding

The per-cluster sections above use the pipeline's own pose clusters
(`scripts/pose_cluster_anchor.py`: rigid DCL4-anchor Kabsch + hierarchical
RMSD on the partner chains -- 2 clusters). This section is a **manual
override**: take the *same* PCA embedding those pose clusters were cut from
(`results/rna_ds_dcl4_drb2_drb4/pose_clusters.csv`'s `pc1`/`pc2`, the same
embedding `notebooks/pose_clustering.ipynb` plots) and split it into **four
quadrants** with a vertical line at `pc1 = 0` and a horizontal line at
`pc2 = 0` -- no model fitting, just the sign of each coordinate. Then re-run
the per-couple domain analysis stratified by those 4 groups.

Quadrants (edit `PC1_SPLIT` / `PC2_SPLIT` in the next cell to move the
crossing point):

| label | region |
|---|---|
| `Q1` | `pc1 >= 0`, `pc2 >= 0` |
| `Q2` | `pc1 < 0`, `pc2 >= 0` |
| `Q3` | `pc1 < 0`, `pc2 < 0` |
| `Q4` | `pc1 >= 0`, `pc2 < 0` |

The split is assigned on **all** models in `pose_clusters.csv` (every
backend); the domain heatmaps below then only see this notebook's
energy-filtered subset (AlphaFold3 + OpenFold3). The first cell assigns the
quadrants, plots the `pc1`/`pc2` scatter with the two crossing lines, and
cross-tabs the 4 quadrants against the pipeline's 2 pose clusters and the
backends -- **so the split can be eyeballed** before reading the per-quadrant
heatmaps. Figures and CSVs land in `figures/domain_analysis/per_quadrant/`.

In [18]:
## Manual pose clustering: 4 quadrants of the pipeline's PCA embedding.
## Split pose_clusters.csv's pc1/pc2 (pose_cluster_anchor.py's anchor-Kabsch + partner-RMSF
## PCA embedding, the same one notebooks/pose_clustering.ipynb plots) with a vertical line at
## pc1=0 and a horizontal line at pc2=0 -- four quadrants, no model fitting.
COMPLEX_NAME = RESULTS_DIR.name
CLUSTER_PALETTE = px.colors.qualitative.Set1
BACKEND_SYMBOLS = {"alphafold3": "circle", "boltz": "square", "chai1": "diamond",
                   "openfold3": "triangle-up", "protenix": "x", "rosettafold3": "cross"}

PC1_SPLIT, PC2_SPLIT = 0.0, 0.0   # the crossing lines; edit to move the quadrant boundaries

pose = pd.read_csv(RESULTS_DIR / "pose_clusters.csv")
if "pc1" not in pose.columns:
    raise ValueError(f"{RESULTS_DIR / 'pose_clusters.csv'} has no pc1/pc2 columns -- re-run "
                     "workflows/postprocessing/Snakefile's pose_cluster rule "
                     "(needs scripts/pose_cluster_anchor.py 2026-08-24 or later).")
print(f"[{COMPLEX_NAME}] pose_clusters.csv: {len(pose)} models, "
      f"{pose['backend'].value_counts().to_dict()}")

def quadrant_label(pc1, pc2):
    right, top = pc1 >= PC1_SPLIT, pc2 >= PC2_SPLIT
    if right and top:         return "Q1 (pc1>=0, pc2>=0)"
    if not right and top:     return "Q2 (pc1<0, pc2>=0)"
    if not right and not top: return "Q3 (pc1<0, pc2<0)"
    return "Q4 (pc1>=0, pc2<0)"

pose["quadrant"] = [quadrant_label(a, b) for a, b in zip(pose["pc1"], pose["pc2"])]
QUADRANTS = sorted(pose["quadrant"].unique(), key=str)

print(f"\nquadrant split at pc1={PC1_SPLIT}, pc2={PC2_SPLIT}")
print("\nmodels per quadrant (all pose_clusters.csv rows):")
print(pose["quadrant"].value_counts().sort_index().rename("n_models").to_frame())
print("\nquadrant x pipeline pose cluster:")
print(pd.crosstab(pose["quadrant"], pose["cluster"]).to_string())
print("\nquadrant x backend:")
print(pd.crosstab(pose["quadrant"], pose["backend"]).to_string())

# --- visualise the quadrant split (same embedding as pose_clustering.ipynb's plot_pca) ---
hover_cols = ["backend", "seed", "sample_index", "cluster", "ptm", "iptm", "ranking_score"]
fig = go.Figure()
for i, cat in enumerate(QUADRANTS):
    sub = pose[pose["quadrant"] == cat]
    for backend in sorted(sub["backend"].unique()):
        bsub = sub[sub["backend"] == backend]
        fig.add_trace(go.Scatter(
            x=bsub["pc1"], y=bsub["pc2"], mode="markers",
            marker=dict(size=7, opacity=0.75,
                        color=CLUSTER_PALETTE[i % len(CLUSTER_PALETTE)],
                        symbol=BACKEND_SYMBOLS.get(backend, "circle"),
                        line=dict(width=0.5, color="white")),
            name=f"{cat} / {backend}", customdata=bsub[hover_cols],
            hovertemplate="<br>".join(f"{c}: %{{customdata[{j}]}}" for j, c in enumerate(hover_cols))
                          + "<extra></extra>"))
fig.add_vline(x=PC1_SPLIT, line=dict(color="black", width=2, dash="dash"))
fig.add_hline(y=PC2_SPLIT, line=dict(color="black", width=2, dash="dash"))
fig.update_layout(title=f"{COMPLEX_NAME}: manual 4-quadrant split of pose PCA (pc1/pc2)",
                  xaxis_title="PC1", yaxis_title="PC2", template=TEMPLATE,
                  height=620, width=820, legend=dict(font=dict(size=9)))
save_fig(fig, f"{COMPLEX_NAME}_pose_quadrant_clusters.html", "per_quadrant")

# --- attach the quadrant label to this notebook's PLIP contact rows, joined on the raw CIF path ---
sel_key = pd.read_csv(RESULTS_DIR / "selected_models.csv")
sel_key["fname"] = sel_key["staged_cif"].apply(lambda p: Path(p).stem)
sel_key["cif_key"] = sel_key["source_cif"].str.replace(r"^results/abcfold/", "", regex=True)
sel_key = sel_key.merge(pose[["cif_path", "quadrant"]].rename(columns={"cif_path": "cif_key"}),
                        on="cif_key", how="left")
n_nolabel = sel_key["quadrant"].isna().sum()
if n_nolabel:
    print(f"\nWARNING: {n_nolabel}/{len(sel_key)} selected models had no pose_clusters.csv match")

df = df.drop(columns=[col for col in ["quadrant"] if col in df.columns])
df = df.merge(sel_key[["fname", "cluster", "quadrant"]].drop_duplicates(),
              on=["fname", "cluster"], how="left", validate="many_to_one")

quad_n_models = df.dropna(subset=["quadrant"]).groupby("quadrant")["fname"].nunique()
QUADRANTS_DF = sorted(quad_n_models.index.tolist(), key=str)
print("\nmodels per quadrant (this notebook's energy-filtered set):")
print(quad_n_models.rename("n_models").to_frame())
print(f"\ncontact rows with no quadrant label (expected 0): {df['quadrant'].isna().sum()}")

[rna_ds_dcl4_drb2_drb4_synthtmpl_01] pose_clusters.csv: 190 models, {'alphafold3': 100, 'openfold3': 90}

quadrant split at pc1=0.0, pc2=0.0

models per quadrant (all pose_clusters.csv rows):
                     n_models
quadrant                     
Q1 (pc1>=0, pc2>=0)        53
Q2 (pc1<0, pc2>=0)         19
Q3 (pc1<0, pc2<0)          18
Q4 (pc1>=0, pc2<0)        100

quadrant x pipeline pose cluster:
cluster                1   2
quadrant                    
Q1 (pc1>=0, pc2>=0)   53   0
Q2 (pc1<0, pc2>=0)     2  17
Q3 (pc1<0, pc2<0)      7  11
Q4 (pc1>=0, pc2<0)   100   0

quadrant x backend:
backend              alphafold3  openfold3
quadrant                                  
Q1 (pc1>=0, pc2>=0)          42         11
Q2 (pc1<0, pc2>=0)           15          4
Q3 (pc1<0, pc2<0)             4         14
Q4 (pc1>=0, pc2<0)           39         61


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/rna_ds_dcl4_drb2_drb4_synthtmpl_01_pose_quadrant_clusters.html



models per quadrant (this notebook's energy-filtered set):
                     n_models
quadrant                     
Q1 (pc1>=0, pc2>=0)        50
Q2 (pc1<0, pc2>=0)         17
Q3 (pc1<0, pc2<0)          17
Q4 (pc1>=0, pc2<0)         87

contact rows with no quadrant label (expected 0): 0


In [19]:
## Domain contact heatmaps, one panel per quadrant
## (same layout as the "Per-backend breakdown" panel, split by quadrant instead)
def quadrant_panel_for_couple(ligand_chain):
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    per_quad = {}
    for q in QUADRANTS_DF:
        res = couple_heatmap_matrix(df[df["quadrant"] == q], ligand_chain, quad_n_models[q])
        if res is not None:
            per_quad[q] = res
    if not per_quad:
        print(f"No DCL4-{ligand_name} contacts for any quadrant -- skipping panel.")
        return

    present = list(per_quad.keys())
    zmax = max(rate.values.max() for _, _, rate in per_quad.values())
    ncols = min(2, len(present))
    nrows = -(-len(present) // ncols)

    fig = make_subplots(
        rows=nrows, cols=ncols,
        subplot_titles=[f"{q} (n={quad_n_models[q]})" for q in present],
        shared_yaxes=True,
    )
    for i, q in enumerate(present):
        labels, y_labels, rate = per_quad[q]
        row, col = i // ncols + 1, i % ncols + 1
        fig.add_trace(
            go.Heatmap(
                z=rate.values, x=labels, y=y_labels, colorscale="YlOrRd",
                zmin=0, zmax=zmax, showscale=(q == present[-1]),
                text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in rate.values],
                texttemplate="%{text}", textfont=dict(size=8),
                hovertemplate=f"{q}<br>DCL4: %{{y}}<br>{ligand_name}: %{{x}}"
                              "<br>%{z:.3f} contacts/model<extra></extra>",
                colorbar=dict(title="Mean<br>contacts/<br>model"),
            ),
            row=row, col=col,
        )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        title=f"DCL4 x {ligand_name} domain contact pairs by pose-PCA quadrant",
        template=TEMPLATE, width=max(760, 420 * ncols), height=max(500, 320 * nrows),
    )
    save_fig(fig, f"dcl4_{ligand_name.lower()}_heatmap_by_quadrant.html", "per_quadrant")

for c in COUPLES:
    quadrant_panel_for_couple(c)

Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_drb2_heatmap_by_quadrant.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_drb4_heatmap_by_quadrant.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_rna(d)_heatmap_by_quadrant.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_rna(e)_heatmap_by_quadrant.html


In [20]:
## Interaction-type breakdown per quadrant
dfq = df.dropna(subset=["quadrant"])

print("Interaction type by quadrant (contact rows):")
display(pd.crosstab(dfq["quadrant"], dfq["interaction_type"]))

per_quad_itype = (
    dfq.groupby(["quadrant", "interaction_type"], observed=True).size()
       .div(quad_n_models, level="quadrant")
       .rename("contacts_per_model").reset_index()
)
fig = px.bar(per_quad_itype, x="quadrant", y="contacts_per_model", color="interaction_type",
             color_discrete_sequence=px.colors.qualitative.Set1, template=TEMPLATE,
             category_orders={"quadrant": QUADRANTS_DF},
             title="DCL4 contacts per model, by interaction type and pose-PCA quadrant")
fig.update_layout(width=760, height=460, xaxis_title="quadrant", yaxis_title="contacts / model")
save_fig(fig, "dcl4_interaction_types_by_quadrant.html", "per_quadrant")

Interaction type by quadrant (contact rows):


interaction_type,hydrogen_bonds,hydrophobic_interactions,pi-cation_interactions,pi-stacking,salt_bridges
quadrant,,,,,
"Q1 (pc1>=0, pc2>=0)",6370,1491,91,14,3225
"Q2 (pc1<0, pc2>=0)",2162,460,30,1,1163
"Q3 (pc1<0, pc2<0)",2433,803,39,12,1339
"Q4 (pc1>=0, pc2<0)",11842,3817,197,49,6532


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_interaction_types_by_quadrant.html


In [21]:
## Export per-quadrant / per-couple domain contact-rate tables
## (copy of the "Export per-cluster tables" cell, with pose `cluster` -> `quadrant`)
for q in QUADRANTS_DF:
    sub_q = df[df["quadrant"] == q]
    n_q = quad_n_models[q]
    tag = q.split()[0].lower()   # "Q1 (pc1>=0, pc2>=0)" -> "q1"
    for lig in COUPLES:
        ct = couple_heatmap_matrix(sub_q, lig, n_q)
        if ct is None:
            continue
        _, _, rate = ct
        out_csv = out_path(
            "per_quadrant",
            f"dcl4_{LIGAND_CHAIN_NAMES[lig].lower()}_domain_pair_rate_{tag}.csv",
        )
        rate.to_csv(out_csv)
        print(f"Saved: {out_csv}")

Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_drb2_domain_pair_rate_q1.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_drb4_domain_pair_rate_q1.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_rna(d)_domain_pair_rate_q1.csv


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_rna(e)_domain_pair_rate_q1.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_drb2_domain_pair_rate_q2.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_drb4_domain_pair_rate_q2.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_rna(d)_domain_pair_rate_q2.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_rna(e)_domain_pair_rate_q2.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_drb2_domain_pair_rate_q3.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_drb4_domain_pair_rate_q3.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_rna(d)_domain_pair_rate_q3.csv


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_rna(e)_domain_pair_rate_q3.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_drb2_domain_pair_rate_q4.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_drb4_domain_pair_rate_q4.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_rna(d)_domain_pair_rate_q4.csv
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/per_quadrant/dcl4_rna(e)_domain_pair_rate_q4.csv


---

# DRB2 x DRB4 interface -- dedicated PLIP pass

Neither PLIP pass loaded above sees the **DRB2-DRB4** interface: the main `plip:`
pass puts DRB2 (B) and DRB4 (C) *both* in the ligand group (receptor = DCL4),
and PLIP never reports ligand-vs-ligand contacts; the RNA-ligand pass is
receptor-restricted the other way. `configs/rna_ds_dcl4_drb2_drb4.yaml` now has a
third `plip_drb2_drb4:` block -- receptor = DRB2, ligand = DRB4, exactly as
`configs/drb2_drb4.yaml` / `configs/rna_ds_drb2_drb4.yaml` do for this couple in
their DCL4-free complexes -- aggregated to `all_selected_summary_drb2_drb4.csv`.

This section loads it as a standalone **DRB2 (receptor) x DRB4 (ligand)** couple:
same energy filter (`good_pairs`) and same pose-PCA quadrants as the rest of the
notebook, but its own receptor axis (DRB2 domains, not DCL4). Figures land in
`figures/domain_analysis/drb2_drb4/`.


In [22]:
## DRB2 x DRB4 -- load the dedicated pass, energy-filter, map both sides to domains
bc_csv = RESULTS_DIR / "all_selected_summary_drb2_drb4.csv"
if not bc_csv.exists():
    raise FileNotFoundError(
        f"{bc_csv.name} not found -- run the plip_drb2_drb4 postprocessing pass:\n"
        f"  snakemake -s workflows/postprocessing/Snakefile --cores 4 --use-conda \\\n"
        f"    {bc_csv}")

df_bc = pd.read_csv(bc_csv).rename(columns={"replica": "cluster", "model": "fname"})
df_bc["cluster"] = df_bc["cluster"].astype(int)
df_bc = df_bc.merge(sel, on=["fname", "cluster"], how="left", validate="many_to_one")

# same energy filter as the DCL4 couples -- reuse the good (cluster, fname) set
bc_keys = pd.MultiIndex.from_arrays([df_bc["cluster"], df_bc["fname"]])
n_bc_before = df_bc.groupby(["cluster", "fname"]).ngroups
df_bc = df_bc[bc_keys.isin(good_pairs)].copy()
n_bc_after = df_bc.groupby(["cluster", "fname"]).ngroups
print(f"{bc_csv.name}: {len(df_bc)} contact rows, {n_bc_after}/{n_bc_before} model(s) after the energy filter")
print("\nreschain x reschain_lig (expect only B x C):")
print(df_bc.groupby(["reschain", "reschain_lig"]).size().rename("rows").to_frame())

# receptor = DRB2 (chain B, continuous resnr), ligand = DRB4 (chain C) -- subtract the
# same per-chain offsets the DCL4-couple cell verified, then map to each protein's domains.
df_bc["drb2_resnr"] = df_bc["resnr"] - CHAIN_OFFSET["B"]       # 1702-2135 -> 1-434
df_bc["drb4_resnr"] = df_bc["resnr_lig"] - CHAIN_OFFSET["C"]   # 2136-2490 -> 1-355
for name, col, hi in [("DRB2", "drb2_resnr", 434), ("DRB4", "drb4_resnr", 355)]:
    bad = int((~df_bc[col].between(1, hi)).sum())
    print(f"{name}: {bad} rows outside [1,{hi}] after offset -- re-check CHAIN_OFFSET" if bad
          else f"{name}: all {len(df_bc)} rows land in [1,{hi}] (offset OK)")

df_bc["drb2_domain"] = LIGAND_MAPPERS["B"](df_bc["drb2_resnr"])
df_bc["drb4_domain"] = LIGAND_MAPPERS["C"](df_bc["drb4_resnr"])
n_unmapped = df_bc[["drb2_domain", "drb4_domain"]].isna().any(axis=1).sum()
print(f"\nrows with an unmapped domain on either side: {n_unmapped} ({100*n_unmapped/len(df_bc):.1f}%)")
df_bc[["drb2_resnr", "drb2_domain", "drb4_resnr", "drb4_domain", "interaction_type"]].head(5)

all_selected_summary_drb2_drb4.csv: 5774 contact rows, 80/88 model(s) after the energy filter

reschain x reschain_lig (expect only B x C):
                       rows
reschain reschain_lig      
B        C             5774
DRB2: all 5774 rows land in [1,434] (offset OK)
DRB4: all 5774 rows land in [1,355] (offset OK)

rows with an unmapped domain on either side: 21 (0.4%)


,drb2_resnr,drb2_domain,drb4_resnr,drb4_domain,interaction_type
0,185,dsRBD2,146,dsRBD2,hydrophobic_interactions
1,384,disordered,149,dsRBD2,hydrophobic_interactions
2,392,disordered,73,dsRBD1,hydrophobic_interactions
3,396,disordered,74,linker,hydrophobic_interactions
4,423,disordered,184,disordered,hydrophobic_interactions


In [23]:
## DRB2 x DRB4 -- pooled domain-pair heatmap + interaction-type breakdown
DRB2_LABELS = [l for l, _, _ in DRB2_DOMAINS]
DRB4_LABELS = [l for l, _, _ in DRB4_DOMAINS]

def drb2_drb4_domain_matrix(data, n_models_norm):
    sub = data.dropna(subset=["drb2_domain", "drb4_domain"])
    if sub.empty:
        return None
    ct = (sub.groupby(["drb2_domain", "drb4_domain"], observed=True).size()
          .unstack(fill_value=0).reindex(index=DRB2_LABELS, columns=DRB4_LABELS, fill_value=0))
    return ct / n_models_norm if n_models_norm else ct

def drb2_drb4_heatmap(rate, title, filename, subdir="drb2_drb4"):
    fig = go.Figure(go.Heatmap(
        z=rate.values, x=DRB4_LABELS, y=DRB2_LABELS, colorscale="YlOrRd",
        text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in rate.values],
        texttemplate="%{text}", textfont=dict(size=9),
        hovertemplate="DRB2 domain: %{y}<br>DRB4 domain: %{x}<br>%{z:.3f} contacts/model<extra></extra>",
        colorbar=dict(title="Mean<br>contacts/<br>model")))
    fig.update_layout(title=title, xaxis_title="DRB4 domain", yaxis_title="DRB2 domain",
                      yaxis=dict(autorange="reversed"), template=TEMPLATE,
                      width=max(520, len(DRB4_LABELS) * 110), height=max(420, len(DRB2_LABELS) * 74))
    save_fig(fig, filename, subdir)

n_bc = df_bc.groupby(["cluster", "fname"]).ngroups
rate_all = drb2_drb4_domain_matrix(df_bc, n_bc)
if rate_all is None:
    print("No DRB2-DRB4 domain-mapped contacts -- nothing to plot.")
else:
    drb2_drb4_heatmap(rate_all,
        f"DRB2 x DRB4 -- all clusters, all backends pooled (n={n_bc} models)",
        "drb2_drb4_heatmap_pooled.html")
    rate_all.to_csv(out_path("drb2_drb4", "drb2_drb4_domain_pair_rate_pooled.csv"))

print("Interaction types (DRB2 x DRB4, pooled):")
display(df_bc["interaction_type"].value_counts().rename("count").to_frame())

per_backend_itype = (
    df_bc.dropna(subset=["backend"]).groupby(["backend", "interaction_type"], observed=True).size()
         .div(df_bc.groupby("backend")["fname"].nunique(), level="backend")
         .rename("contacts_per_model").reset_index()
)
fig = px.bar(per_backend_itype, x="backend", y="contacts_per_model", color="interaction_type",
             color_discrete_sequence=px.colors.qualitative.Set1, template=TEMPLATE,
             title="DRB2 x DRB4 contacts per model, by interaction type and backend")
fig.update_layout(width=720, height=440, xaxis_title="", yaxis_title="contacts / model")
save_fig(fig, "drb2_drb4_interaction_types_by_backend.html", "drb2_drb4")

Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/drb2_drb4/drb2_drb4_heatmap_pooled.html


Interaction types (DRB2 x DRB4, pooled):


,count
interaction_type,
hydrogen_bonds,3238
hydrophobic_interactions,2148
salt_bridges,349
pi-cation_interactions,33
pi-stacking,6


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/drb2_drb4/drb2_drb4_interaction_types_by_backend.html


In [24]:
## DRB2 x DRB4 -- domain-pair heatmap per pose-PCA quadrant
df_bc = df_bc.drop(columns=[c for c in ["quadrant"] if c in df_bc.columns])
df_bc = df_bc.merge(sel_key[["fname", "cluster", "quadrant"]].drop_duplicates(),
                    on=["fname", "cluster"], how="left", validate="many_to_one")
bc_quad_n = df_bc.dropna(subset=["quadrant"]).groupby("quadrant")["fname"].nunique()

mats = {}
for q in QUADRANTS_DF:
    if q not in bc_quad_n.index:
        continue
    m = drb2_drb4_domain_matrix(df_bc[df_bc["quadrant"] == q], bc_quad_n[q])
    if m is not None:
        mats[q] = m

if not mats:
    print("No DRB2-DRB4 contacts in any quadrant -- skipping panel.")
else:
    qs = list(mats)
    zmax = max(m.values.max() for m in mats.values())
    ncols = min(2, len(qs))
    nrows = -(-len(qs) // ncols)
    fig = make_subplots(rows=nrows, cols=ncols,
                        subplot_titles=[f"{q} (n={bc_quad_n[q]})" for q in qs], shared_yaxes=True)
    for i, q in enumerate(qs):
        m = mats[q]
        row, col = i // ncols + 1, i % ncols + 1
        fig.add_trace(go.Heatmap(
            z=m.values, x=DRB4_LABELS, y=DRB2_LABELS, colorscale="YlOrRd",
            zmin=0, zmax=zmax, showscale=(q == qs[-1]),
            text=[[f"{v:.2f}" if v > 0 else "" for v in r] for r in m.values],
            texttemplate="%{text}", textfont=dict(size=8),
            hovertemplate=f"{q}<br>DRB2: %{{y}}<br>DRB4: %{{x}}<br>%{{z:.3f}} contacts/model<extra></extra>",
            colorbar=dict(title="Mean<br>contacts/<br>model")), row=row, col=col)
        m.to_csv(out_path("drb2_drb4", f"drb2_drb4_domain_pair_rate_{q.split()[0].lower()}.csv"))
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(title="DRB2 x DRB4 domain contact pairs by pose-PCA quadrant",
                      template=TEMPLATE, width=max(760, 420 * ncols), height=max(500, 320 * nrows))
    save_fig(fig, "drb2_drb4_heatmap_by_quadrant.html", "drb2_drb4")

Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/drb2_drb4/drb2_drb4_heatmap_by_quadrant.html


---

# Contact heatmaps for an over-folding-filter survivor

Unlike the baseline `rna_ds_dcl4_drb2_drb4` (which leaves **1** survivor),
the quick-look over-folding filter here keeps **~38 models, all AlphaFold3**
(DRB2 disordered tail <=20% helix and <12-res run; DRB4 partially rescued).
The list is `results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/`
`overfolding_survivors_quicklook.tsv` (cleanest first).

This section zooms the whole contact analysis onto **one** survivor so its
interface can be read residue by residue -- domain level and residue level,
for every DCL4 couple plus DRB2 x DRB4. Values are **raw contact counts** for
that single structure (no per-model averaging).

The survivor id is read from `filtered_models.csv` if the canonical
`notebooks/rna_complexes_energy_and_overfolding_filter.ipynb` has been re-run
for this complex, else from `overfolding_survivors_quicklook.tsv`. Edit
`SURVIVOR` in the next cell to point at another model.

**Model-specific clashes.** `CLASH_EXCLUSIONS` starts **empty** for this run
-- the baseline's RNA(E) 47-50 x DCL4 PAZ clash is baseline-survivor-specific.
Inspect the chosen survivor's DCL4 x RNA heatmaps below and add
`(chain, lig_lo, lig_hi, dcl4_lo, dcl4_hi)` tuples if an implausible contact
block shows up. Figures land in
`figures/domain_analysis/overfolding_survivor/`.


In [25]:
# --- pick the surviving model ----------------------------------------------
_filt = (RESULTS_DIR / "figures" / "domain_analysis" / "energy_overfolding_filter"
         / "filtered_models.csv")
if _filt.exists():
    _k = pd.read_csv(_filt)
    _k = _k[_k["keep"].astype(bool)]
    SURVIVOR = (_k["fname"].iloc[0], int(_k["cluster"].iloc[0]))
    if len(_k) != 1:
        print(f"NOTE: {_filt.name} lists {len(_k)} survivors; using the first.")
else:
    _ql = RESULTS_DIR / "overfolding_survivors_quicklook.tsv"
    if _ql.exists():
        _q = pd.read_csv(_ql, sep="\t")
        SURVIVOR = (_q["fname"].iloc[0], int(_q["cluster"].iloc[0]))
        print(f"{_filt.name} not found -- using cleanest row of {_ql.name}: {SURVIVOR[0]}")
    else:
        raise FileNotFoundError(
            f"neither {_filt} nor {_ql} exists -- re-run the over-folding filter "
            f"notebook or scripts/select_synthetic_templates.py's DSSP step first")
SURV_FNAME, SURV_CLUSTER = SURVIVOR
SURV_SUBDIR = "overfolding_survivor"
print(f"survivor model: {SURV_FNAME}  (pose cluster {SURV_CLUSTER})")

# --- model-specific clash rows to drop ------------------------------------
# (ligand_chain, lig_lo, lig_hi, dcl4_lo, dcl4_hi) inclusive; resnr_lig is the
# per-chain 1-based numbering after the CHAIN_OFFSET fix. RNA(E) = chain E.
# baseline had ("E", 47, 50, 974, 991) here; start empty for this run and
# add tuples per the survivor actually chosen above (see the section markdown).
CLASH_EXCLUSIONS = []

def clash_mask(data, exclusions=CLASH_EXCLUSIONS):
    m = pd.Series(False, index=data.index)
    for lig_chain, l_lo, l_hi, d_lo, d_hi in exclusions:
        m |= ((data["reschain_lig"] == lig_chain)
              & data["resnr_lig"].between(l_lo, l_hi)
              & data["resnr"].between(d_lo, d_hi))
    return m

def drop_clash_rows(data, exclusions=CLASH_EXCLUSIONS, verbose=True):
    m = clash_mask(data, exclusions)
    if verbose and m.any():
        print(f"  dropped {int(m.sum())} clash row(s)")
    return data.loc[~m].copy()

# --- subset to the survivor (DCL4 couples) --------------------------------
df_surv = df[(df["fname"] == SURV_FNAME) & (df["cluster"] == SURV_CLUSTER)].copy()
print(f"\n{len(df_surv)} DCL4-couple contact rows for this model:")
display(df_surv.groupby("reschain_lig").size()
        .rename(index=LIGAND_CHAIN_NAMES).rename("rows").to_frame())

_cm = clash_mask(df_surv)
print(f"\nclash rows removed ({int(_cm.sum())}):")
display(df_surv.loc[_cm, ["resnr", "restype", "resnr_lig", "restype_lig", "dist", "interaction_type"]]
        .sort_values(["resnr", "resnr_lig"]).reset_index(drop=True))

df_surv_nc = df_surv.loc[~_cm].copy()
print(f"{len(df_surv)} -> {len(df_surv_nc)} DCL4-couple rows after clash removal")

filtered_models.csv not found -- using cleanest row of overfolding_survivors_quicklook.tsv: rank_01_alphafold3_seed3_sample0.0
survivor model: rank_01_alphafold3_seed3_sample0.0  (pose cluster 2)

222 DCL4-couple contact rows for this model:


,rows
reschain_lig,
DRB2,30
DRB4,39
RNA(D),88
RNA(E),65



clash rows removed (0):


,resnr,restype,resnr_lig,restype_lig,dist,interaction_type


222 -> 222 DCL4-couple rows after clash removal


In [26]:
# --- Domain level: one heatmap per DCL4 couple for the survivor -----------
# n=1 model, so heatmap values are raw contact counts. -> overfolding_survivor/
def _slug(ch):
    return LIGAND_CHAIN_NAMES[ch].lower().replace("(", "_").replace(")", "")

dom_surv = {}
for ch in COUPLES:
    dom_surv[ch] = couple_heatmap(
        df_surv_nc, ch, 1,
        f"{SURV_FNAME}  --  DCL4 x {LIGAND_CHAIN_NAMES[ch]} domain contacts "
        f"(cluster {SURV_CLUSTER}, raw counts, clash-filtered)",
        f"survivor_dcl4_{_slug(ch)}_domain.html",
        subdir=SURV_SUBDIR,
    )

Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_dcl4_drb2_domain.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_dcl4_drb4_domain.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_dcl4_rna_d_domain.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_dcl4_rna_e_domain.html


In [27]:
def residue_contact_heatmap(data, ligand_chain, n_models_norm, *,
                            min_total_contacts=None, dcl4_range=None, lig_range=None,
                            value_label="contacts/model", title=None, filename=None,
                            subdir="residue_contacts"):
    """DCL4 residue x partner residue/nt contact heatmap.

    n_models_norm      : divide raw counts by this. Pass 1 for a single model
                         (values are raw counts; set value_label="contacts").
    min_total_contacts : drop any DCL4 res / partner res whose summed raw count
                         is below this (default max(3, 3% of n_models_norm);
                         pass 1 for a single model).
    dcl4_range / lig_range : (lo, hi) inclusive axis restrictions.
    """
    is_rna = ligand_chain in ("D", "E")
    ligand_name = LIGAND_CHAIN_NAMES[ligand_chain]
    slug = ligand_name.lower().replace("(", "_").replace(")", "")
    if min_total_contacts is None:
        min_total_contacts = max(3, round(0.03 * (n_models_norm or 0)))

    sub = data[data["reschain_lig"] == ligand_chain].dropna(subset=["resnr", "resnr_lig"]).copy()
    sub[["resnr", "resnr_lig"]] = sub[["resnr", "resnr_lig"]].astype(int)
    if dcl4_range:
        sub = sub[sub["resnr"].between(*dcl4_range)]
    if lig_range:
        sub = sub[sub["resnr_lig"].between(*lig_range)]
    if sub.empty:
        print(f"No DCL4-{ligand_name} contacts in this subset -- skipping.")
        return None

    ct = (sub.groupby(["resnr", "resnr_lig"]).size().unstack(fill_value=0)
             .sort_index().sort_index(axis=1))
    ct = ct.loc[ct.sum(axis=1) >= min_total_contacts, ct.sum(axis=0) >= min_total_contacts]
    if ct.empty:
        print(f"DCL4-{ligand_name}: nothing left at min_total_contacts={min_total_contacts}.")
        return None
    rate = ct / n_models_norm if n_models_norm else ct
    zfmt = ".0f" if n_models_norm == 1 else ".3f"

    y_res, x_res = ct.index.tolist(), ct.columns.tolist()
    dom_y = dcl4_mapper(pd.Series(y_res, index=y_res)).tolist()
    top_itype = (sub.groupby(["resnr", "resnr_lig"])["interaction_type"]
                 .agg(lambda s: s.value_counts().idxmax())
                 .unstack().reindex(index=y_res, columns=x_res))
    customdata = np.dstack([np.tile(np.array(dom_y, dtype=object)[:, None], (1, len(x_res))),
                            top_itype.to_numpy(dtype=object)])

    fig = go.Figure(go.Heatmap(
        z=rate.values, x=[str(p) for p in x_res], y=[str(p) for p in y_res],
        colorscale="YlOrRd", customdata=customdata,
        hovertemplate=(f"DCL4 residue: %{{y}} (%{{customdata[0]}})<br>"
                       f"{ligand_name} {'nt' if is_rna else 'residue'}: %{{x}}<br>"
                       f"%{{z:{zfmt}}} {value_label}<br>top interaction: %{{customdata[1]}}"
                       "<extra></extra>"),
        colorbar=dict(title=value_label.replace("/", "/<br>")),
    ))
    fig.update_layout(
        title=title or (f"DCL4 x {ligand_name} -- residue-resolution {value_label} "
                        f"(min total {min_total_contacts})"),
        xaxis_title=f"{ligand_name} {'nt position' if is_rna else 'residue'}",
        yaxis_title="DCL4 residue",
        xaxis=dict(type="category", tickfont=dict(size=7)),
        yaxis=dict(type="category", autorange="reversed", tickfont=dict(size=7)),
        template=TEMPLATE,
        width=min(1700, max(560, len(x_res) * 16 + 220)),
        height=min(2400, max(420, len(y_res) * 16 + 160)),
    )
    save_fig(fig, filename or f"dcl4_{slug}_residue_heatmap.html", subdir)
    return ct

# --- Residue level: one heatmap per DCL4 couple for the survivor ----------
res_surv = {}
for ch in COUPLES:
    res_surv[ch] = residue_contact_heatmap(
        df_surv_nc, ch, 1, min_total_contacts=1, value_label="contacts",
        title=(f"{SURV_FNAME}  --  DCL4 x {LIGAND_CHAIN_NAMES[ch]} residue contacts "
               f"(cluster {SURV_CLUSTER}, clash-filtered)"),
        filename=f"survivor_dcl4_{_slug(ch)}_residue.html",
        subdir=SURV_SUBDIR,
    )

Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_dcl4_drb2_residue.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_dcl4_drb4_residue.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_dcl4_rna_d_residue.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_dcl4_rna_e_residue.html


In [28]:
# --- DRB2 x DRB4 interface of the survivor (dedicated plip_drb2_drb4 pass) --
bc_surv = df_bc[(df_bc["fname"] == SURV_FNAME) & (df_bc["cluster"] == SURV_CLUSTER)].copy()
if bc_surv.empty:
    print(f"No DRB2 x DRB4 contacts recorded for {SURV_FNAME} "
          f"in all_selected_summary_drb2_drb4.csv.")
else:
    print(f"{len(bc_surv)} DRB2 x DRB4 contact rows for this model")

    rate_bc = drb2_drb4_domain_matrix(bc_surv, 1)   # n=1 -> raw counts
    if rate_bc is not None:
        drb2_drb4_heatmap(
            rate_bc,
            f"{SURV_FNAME}  --  DRB2 x DRB4 domain contacts (cluster {SURV_CLUSTER}, raw counts)",
            "survivor_drb2_drb4_domain.html", subdir=SURV_SUBDIR)

    ct = (bc_surv.dropna(subset=["drb2_resnr", "drb4_resnr"])
          .assign(drb2_resnr=lambda d: d["drb2_resnr"].astype(int),
                  drb4_resnr=lambda d: d["drb4_resnr"].astype(int))
          .groupby(["drb2_resnr", "drb4_resnr"]).size().unstack(fill_value=0)
          .sort_index().sort_index(axis=1))
    if not ct.empty:
        top_it = (bc_surv.groupby(["drb2_resnr", "drb4_resnr"])["interaction_type"]
                  .agg(lambda s: s.value_counts().idxmax()).unstack()
                  .reindex(index=ct.index, columns=ct.columns))
        db2_dom = LIGAND_MAPPERS["B"](pd.Series(ct.index, index=ct.index)).tolist()
        fig = go.Figure(go.Heatmap(
            z=ct.values, x=[str(c) for c in ct.columns], y=[str(i) for i in ct.index],
            colorscale="YlOrRd",
            customdata=np.dstack([np.tile(np.array(db2_dom, dtype=object)[:, None], (1, ct.shape[1])),
                                  top_it.to_numpy(dtype=object)]),
            hovertemplate=("DRB2 residue: %{y} (%{customdata[0]})<br>DRB4 residue: %{x}<br>"
                           "%{z:.0f} contacts<br>top interaction: %{customdata[1]}<extra></extra>"),
            colorbar=dict(title="contacts")))
        fig.update_layout(
            title=f"{SURV_FNAME}  --  DRB2 x DRB4 residue contacts (cluster {SURV_CLUSTER})",
            xaxis_title="DRB4 residue", yaxis_title="DRB2 residue",
            xaxis=dict(type="category", tickfont=dict(size=8)),
            yaxis=dict(type="category", autorange="reversed", tickfont=dict(size=8)),
            template=TEMPLATE,
            width=max(560, ct.shape[1] * 22 + 220), height=max(420, ct.shape[0] * 22 + 160))
        save_fig(fig, "survivor_drb2_drb4_residue.html", SURV_SUBDIR)

8 DRB2 x DRB4 contact rows for this model
Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_drb2_drb4_domain.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_drb2_drb4_residue.html


In [29]:
# --- Zoom on the excised clash: DCL4 PAZ + connector (932-1082) x RNA(E) ---
# clash-filtered (no-op unless CLASH_EXCLUSIONS populated) vs. raw. The removed block is DCL4 977-991 x RNA(E) nt 47-49.
residue_contact_heatmap(
    df_surv_nc, "E", 1, min_total_contacts=1, dcl4_range=(932, 1082), value_label="contacts",
    title=f"{SURV_FNAME}  --  DCL4 PAZ/connector x RNA(E), clash-filtered (no-op unless CLASH_EXCLUSIONS populated)",
    filename="survivor_dcl4_rna_e_PAZ_zoom.html", subdir=SURV_SUBDIR)

residue_contact_heatmap(
    df_surv, "E", 1, min_total_contacts=1, dcl4_range=(932, 1082), value_label="contacts",
    title=f"{SURV_FNAME}  --  DCL4 PAZ/connector x RNA(E), RAW (clash block visible)",
    filename="survivor_dcl4_rna_e_PAZ_zoom_RAW.html", subdir=SURV_SUBDIR + "/_reference")

Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/survivor_dcl4_rna_e_PAZ_zoom.html


Saved: ../results/rna_ds_dcl4_drb2_drb4_synthtmpl_01/figures/domain_analysis/overfolding_survivor/_reference/survivor_dcl4_rna_e_PAZ_zoom_RAW.html


resnr_lig,2,3,9,10
resnr,,,,
966,1,1,0,0
967,1,0,0,0
1025,0,0,0,1
1026,0,0,0,1
1027,0,0,0,1
1029,0,0,1,0
